# BNK BBR vs no-BBR alignment QC

Exploratory quality-control notebook comparing EPI-derived brain-mask alignment for BNK scans processed with and without boundary-based registration (BBR)

Configure inputs with the environment variables `ROSMAP_NOBBR_MASK_GLOB`, `ROSMAP_BBR_MASK_GLOB`


In [ ]:
import os
from pathlib import Path
from math import ceil
from glob import glob
import nibabel as nib
import numpy as np
from nilearn import plotting
from nilearn.image import load_img
from templateflow import api as tflow
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
# ------------------------------------------------------------------
# Inputs
# ------------------------------------------------------------------
# environment variables for glob patterns to find the no-BBR and BBR functional masks in MNI template space
# for the plotting to work, masks must be in MNI space

nobbr_pattern = os.environ.get("ROSMAP_NOBBR_MASK_GLOB")
bbr_pattern = os.environ.get("ROSMAP_BBR_MASK_GLOB")

if not nobbr_pattern or not bbr_pattern:
    raise RuntimeError(
        "Set ROSMAP_NOBBR_MASK_GLOB and ROSMAP_BBR_MASK_GLOB "
        "before running this cell."
    )

mask_files_nobbr = sorted(glob(nobbr_pattern, recursive=True))
mask_files_bbr = sorted(glob(bbr_pattern, recursive=True))

if not mask_files_nobbr:
    raise RuntimeError(f"No no-BBR masks matched: {nobbr_pattern}")
if not mask_files_bbr:
    raise RuntimeError(f"No BBR masks matched: {bbr_pattern}")

print(f"Found {len(mask_files_nobbr):,} no-BBR masks")
print(f"Found {len(mask_files_bbr):,} BBR masks")


In [ ]:
# ------------------------------------------------------------------
# Files and MNI152NLin6Asym reference
# ------------------------------------------------------------------

mni_t1 = load_img(tflow.get(
    "MNI152NLin6Asym",
    resolution=2,
    suffix="T1w",
    desc="brain",
    extension="nii.gz",
))

mni_brain_mask = load_img(tflow.get(
    "MNI152NLin6Asym",
    resolution=2,
    suffix="mask",
    desc="brain",
    extension="nii.gz",
))

mni_brain = mni_brain_mask.get_fdata() > 0.5

In [ ]:
def plot_alignment_epi(
    mni_t1,
    mni_brain_mask,
    mask_files,
    label,
    #title="EPI-derived brain-mask alignment",
    bad_overlap_threshold=0.80,
    figsize=(18, 9),
):
    mask_files = list(mask_files)

    mni_brain = mni_brain_mask.get_fdata() > 0.5
    count = np.zeros(mni_t1.shape, dtype=np.uint16)

    bad_masks = []
    empty_masks = []
    overlap_results = []

    for i, filename in enumerate(mask_files, start=1):
        img = load_img(filename)

        if img.shape != mni_t1.shape:
            raise ValueError(
                f"Shape mismatch for {filename}: "
                f"{img.shape} != {mni_t1.shape}"
            )

        if not np.allclose(img.affine, mni_t1.affine):
            raise ValueError(f"Affine mismatch for {filename}")

        epi = img.get_fdata() > 0.5
        epi_size = np.count_nonzero(epi)

        if epi_size == 0:
            empty_masks.append(filename)
            overlap_results.append((filename, np.nan))
            continue

        count += epi

        coverage = np.count_nonzero(epi & mni_brain) / epi_size
        overlap_results.append((filename, coverage))

        if coverage < bad_overlap_threshold:
            bad_masks.append((filename, coverage, img))

        if i % 100 == 0:
            print(f"Processed {i:,}/{len(mask_files):,}")

    n_valid = len(mask_files) - len(empty_masks)

    if n_valid == 0:
        raise ValueError("All input masks are empty")

    thresholds = {
        "85%": ceil(0.85 * n_valid),
        "95%": ceil(0.95 * n_valid),
        "99%": ceil(0.99 * n_valid),
    }

    group_masks = {
        threshold_name: nib.Nifti1Image(
            (count >= required).astype(np.uint8),
            mni_t1.affine,
            mni_t1.header.copy(),
        )
        for threshold_name, required in thresholds.items()
    }

    styles = {
        "85%": ("blue", 2.5),
        "95%": ("purple", 2.5),
        "99%": ("black", 3.0),
    }

    # Create a larger Matplotlib figure
    fig = plt.figure(figsize=figsize)

    display = plotting.plot_anat(
        mni_t1,
        figure=fig,
        display_mode="ortho",
        cut_coords=(0, -20, 10),
        title=None,
        annotate=True,
        draw_cross=False,
        colorbar=False
    )

    # Use a larger figure-level title
    # fig.suptitle(
    #     (
    #         f"{title}\n"
    #         f"{label}: {len(bad_masks):,} masks with "
    #         f"<{bad_overlap_threshold:.0%} MNI-brain coverage"
    #     ),
    #     fontsize=20,
    #     color="white",
    #     fontweight="bold",
    #     y=0.98,
    # )

    # Individual badly aligned masks
    for _, _, bad_img in bad_masks:
        display.add_contours(
            bad_img,
            levels=[0.5],
            colors=["red"],
            linewidths=0.8,
            transparency=0.6,
        )

    plotted_thresholds = []

    # Group-level contours
    for threshold_name, image in group_masks.items():
        if not np.any(image.get_fdata()):
            print(f"{threshold_name} contour is empty; omitted")
            continue

        color, width = styles[threshold_name]

        display.add_contours(
            image,
            levels=[0.5],
            colors=[color],
            linewidths=width,
        )

        plotted_thresholds.append(threshold_name)

    # Nilearn does not automatically make legends for contours,
    # so construct one using proxy lines.
    legend_handles = []

    if bad_masks:
        legend_handles.append(
            Line2D(
                [0],
                [0],
                color="red",
                linewidth=2,
                alpha=0.6,
                label=(
                    "Individual masks with "
                    f"<{bad_overlap_threshold:.0%} "
                    "MNI-brain coverage"
                ),
            )
        )

    legend_text = {
        "85%": "Voxels present in ≥85% of valid masks",
        "95%": "Voxels present in ≥95% of valid masks",
        "99%": "Voxels present in ≥99% of valid masks",
    }

    for threshold_name in plotted_thresholds:
        color, width = styles[threshold_name]

        legend_handles.append(
            Line2D(
                [0],
                [0],
                color=color,
                linewidth=width,
                label=legend_text[threshold_name],
            )
        )

    # if legend_handles:
    #     fig.legend(
    #         handles=legend_handles,
    #         loc="lower center",
    #         bbox_to_anchor=(0.5, 0.02),
    #         ncol=2,
    #         fontsize=14,
    #         frameon=True,
    #         fancybox=True,
    #         framealpha=0.95,
    #     )

    # Reserve space for title and legend
    fig.subplots_adjust(
        top=0.82,
        bottom=0.20,
        left=0.02,
        right=0.98,
    )

    plotting.show()

    return overlap_results, bad_masks, empty_masks, group_masks

In [ ]:
results_bbr = plot_alignment_epi(
    mni_t1=mni_t1,
    mni_brain_mask=mni_brain_mask,
    mask_files=mask_files_bbr,
    label="BBR",
    bad_overlap_threshold=0.80,
    figsize=(14, 7),
    #title="Overlap of EPI-Derived Brain Masks in MNI Space for BNK site (BBR)",
)

In [ ]:
results_nobbr = plot_alignment_epi(
    mni_t1=mni_t1,
    mni_brain_mask=mni_brain_mask,
    mask_files=mask_files_nobbr,
    label="No BBR",
    bad_overlap_threshold=0.80,
    figsize=(14, 7),
    #title="Overlap of EPI-Derived Brain Masks in MNI Space for BNK site (noBBR)",
)

In [ ]:
overlap_results, bad_masks, empty_masks, group_masks = results_nobbr

worst = sorted(
    [
        (filename, coverage)
        for filename, coverage in overlap_results
        if not np.isnan(coverage)
    ],
    key=lambda item: item[1],
)

for filename, coverage in worst[:20]:
    print(f"{coverage:.1%}  {filename.split('/')[-1]}")